# RQ3 – Effect of Preprocessing

**Research Question:** How do different data preprocessing strategies affect the performance of supervised learning models on the Marketing and Product Performance Dataset?

**Dataset:** [Kaggle Link](https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

import glob, os

# Auto-detect the dataset file anywhere in the Colab/Kaggle environment
_search_paths = [
    '/kaggle/input/marketing-and-product-performance-dataset/',
    '/content/sample_data/',
    '/content/',
    '/content/drive/MyDrive/',
    '.',
]
_extensions = ['*.csv', '*.xlsx', '*.xls']
DATA_PATH = None

for _dir in _search_paths:
    for _ext in _extensions:
        _matches = glob.glob(os.path.join(_dir, '**', _ext), recursive=True) + glob.glob(os.path.join(_dir, _ext))
        _matches = [m for m in _matches if 'marketing' in m.lower() or 'product' in m.lower() or 'performance' in m.lower()]
        if _matches:
            DATA_PATH = _matches[0]
            break
    if DATA_PATH:
        break

# Final fallback: pick any CSV/Excel in /content
if DATA_PATH is None:
    for _ext in _extensions:
        _all = glob.glob(f'/content/**/{_ext}', recursive=True) + glob.glob(f'./**/{_ext}', recursive=True)
        if _all:
            DATA_PATH = _all[0]
            break

if DATA_PATH:
    print(f'Dataset found: {DATA_PATH}')
else:
    raise FileNotFoundError(
        'Dataset not found. Please upload the CSV/Excel file to Colab via:\n'
        '  Files panel (left sidebar) → Upload, then re-run this cell.')
RANDOM_STATE = 42
TEST_SIZE = 0.2

Dataset found: /content/marketing_and_product_performance 2.csv


In [2]:
# ── Load ──────────────────────────────────────────────────────────────────────
try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    df = pd.read_excel(DATA_PATH)

candidates = [c for c in df.columns if any(k in c.lower() for k in ['revenue','sales','sale','income','profit','amount'])]
TARGET = candidates[0] if candidates else df.select_dtypes(include=np.number).var().idxmax()

df_clean = df.dropna(thresh=len(df)*0.5, axis=1).copy()
X_raw = df_clean.drop(columns=[TARGET]).copy()
y = df_clean[TARGET]

cat_cols = X_raw.select_dtypes(include='object').columns.tolist()
num_cols = X_raw.select_dtypes(include=np.number).columns.tolist()

def encode_cats(X):
    Xc = X.copy()
    for col in cat_cols:
        if col in Xc.columns:
            Xc[col] = LabelEncoder().fit_transform(Xc[col].astype(str))
    return Xc

MODEL = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)

def evaluate(X_tr, X_te, y_tr, y_te):
    MODEL.fit(X_tr, y_tr)
    p = MODEL.predict(X_te)
    return round(mean_absolute_error(y_te,p),4), round(np.sqrt(mean_squared_error(y_te,p)),4), round(r2_score(y_te,p),4)

In [3]:
# ── Strategy 1: Raw Data (drop missing rows) ─────────────────────────────────
X1 = encode_cats(X_raw).fillna(0)
Xtr1, Xte1, ytr1, yte1 = train_test_split(X1, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
m1 = evaluate(Xtr1, Xte1, ytr1, yte1)
print('Raw data:', m1)

Raw data: (25016.1904, np.float64(28964.0431), -0.0288)


In [4]:
# ── Strategy 2: Median imputation only ───────────────────────────────────────
X2 = encode_cats(X_raw)
X2_imp = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X2), columns=X2.columns)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X2_imp, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
m2 = evaluate(Xtr2, Xte2, ytr2, yte2)
print('Imputation only:', m2)

Imputation only: (25016.1904, np.float64(28964.0431), -0.0288)


In [5]:
# ── Strategy 3: Imputation + StandardScaler ───────────────────────────────────
X3_scaled = pd.DataFrame(StandardScaler().fit_transform(X2_imp), columns=X2_imp.columns)
Xtr3, Xte3, ytr3, yte3 = train_test_split(X3_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
m3 = evaluate(Xtr3, Xte3, ytr3, yte3)
print('Imputation + Scaling:', m3)

Imputation + Scaling: (25019.0251, np.float64(28964.8929), -0.0289)


In [6]:
# ── Strategy 4: Full pipeline (impute + scale + outlier capping) ──────────────
X4 = X2_imp.copy()
# Cap outliers at 1.5 IQR
for col in num_cols:
    if col in X4.columns:
        Q1, Q3 = X4[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        X4[col] = X4[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
X4_scaled = pd.DataFrame(StandardScaler().fit_transform(X4), columns=X4.columns)
Xtr4, Xte4, ytr4, yte4 = train_test_split(X4_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
m4 = evaluate(Xtr4, Xte4, ytr4, yte4)
print('Full pipeline:', m4)

Full pipeline: (25019.0251, np.float64(28964.8929), -0.0289)


In [7]:
# ── Save Table ────────────────────────────────────────────────────────────────
strategies = ['Raw Data (fill 0)', 'Median Imputation', 'Imputation + Scaling', 'Full Pipeline']
metrics_list = [m1, m2, m3, m4]

results_df = pd.DataFrame([
    {'Strategy': s, 'MAE': m[0], 'RMSE': m[1], 'R2': m[2]}
    for s, m in zip(strategies, metrics_list)
])
results_df.to_csv('tables/RQ3_preprocessing_impact.csv', index=False)
print('Table saved.')
results_df

Table saved.


,Strategy,MAE,RMSE,R2
0,Raw Data (fill 0),25016.1904,28964.0431,-0.0288
1,Median Imputation,25016.1904,28964.0431,-0.0288
2,Imputation + Scaling,25019.0251,28964.8929,-0.0289
3,Full Pipeline,25019.0251,28964.8929,-0.0289


In [8]:
# ── Figure: Ablation Bar Chart ────────────────────────────────────────────────
x = np.arange(len(results_df))
width = 0.28
colors = ['#2E4057','#048A81','#54C6EB']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (metric, color) in enumerate(zip(['MAE','RMSE','R2'], colors)):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(results_df['Strategy'], fontsize=10, rotation=15, ha='right')
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Figure 3. Performance Gains from Different Preprocessing Strategies\n(Marketing and Product Performance Dataset)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures/RQ3_preprocessing_impact.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


## Summary

**RQ3** compares four preprocessing strategies on a Random Forest model. Results in `tables/RQ3_preprocessing_impact.csv` and `figures/RQ3_preprocessing_impact.pdf`.